In [1]:
import random
import math
import numpy 
import matplotlib.pyplot

In [3]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s  | data %.4f | grad %.4f }" % (n.label , n.data , n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [2]:
class value:
    def __init__(self , data , _children = (), _op = ''  , label = ''):
        self.data = data
        self._op = _op
        self._prev =  set(_children) # It was used by backward to find which nodes created the current node.
        self.grad = 0.0
        self._backward = lambda:None   
        self.label = label

    def __repr__(self):
        return f"Value(data = {self.data})"

    def __add__(self, other):
        other = other if isinstance(other,value) else value(other)
        out = value(self.data + other.data, (self, other), '+') #-> (self,other) -> _childern main diya hai.
# += isliye lagaya hai _backward main taaki apan overlapping bug ko fix kar sake.
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad    
        out._backward = _backward
        return out

    def __radd__(self , other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __mul__(self , other):
        other = other if isinstance(other,value) else value(other)
        out = value(self.data * other.data ,(self , other), "*")
        def _backward():
            self.grad += other.data * out.grad 
            other.grad += self.data * out.grad
        out._backward = _backward      
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * self.data ** (other - 1) * out.grad
        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other,value) else value(other)
        out = value(self.data - other.data, (self, other), '-')
    
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += -1.0 * out.grad
        out._backward = _backward
        return out

    def __rsub__(self, other):
        other = other if isinstance(other, value) else value(other)
        return other - self

    def tanh(self):
        t = math.tanh(self.data)
        out = value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out


    def backward(self):
        self.grad = 1.0

        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)

                for child in v._prev:
                    build_topo(child)

                topo.append(v)

        build_topo(self)

        for node in reversed(topo):
            node._backward()

In [4]:
# Now we have to make an neural network.
class Neuron:
    def __init__(self , no_input):
        self.weights = [value(random.uniform(-1 ,1)) for _ in range(no_input)]
        self.bias = value(random.uniform(-1, 1))

    def __call__(self , x):
        # We wanted to multiply all inputs with their weights and then add bais to them.
        act = sum((wi * xi for wi, xi in zip(self.weights, x)), self.bias)
        out = act.tanh()
        return out         

    def parameters(self):
        return self.weights + [self.bias] 

class Layer:
    def __init__(self, no_input, no_neuron):
        self.neurons = [Neuron(no_input) for _ in range (no_neuron)] # -> number of inpput == number of neuron in next layer

    def __call__(self , x):
        out = [n(x) for n in self.neurons] # -> 
        return out[0] if len(out) == 1 else out
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
class MLP:
    def __init__(self, no_input , no_neuron):
        size = [no_input] + no_neuron
        self.layers = [Layer(size[i] , size[i+1]) for i in range(len(no_neuron))]

    def __call__(self ,x):
        for l in self.layers:
            x = l(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


In [24]:
# dataset:-
xs1 = [
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
]

ys1 = [-1, 1, 1, -1] #-> wanted these outputs.

In [22]:
n = MLP(2 , [4,1])

In [27]:
for k in range(100):
  
  # forward pass
  ypred = [n(x) for x in xs1]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys1, ypred))
  
  # backward pass
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()
  
  # update
  for p in n.parameters():
    p.data += -0.1 * p.grad
  
  print(k, loss.data)

0 2.1076823567585268
1 2.505251096668314
2 1.8789888455123083
3 2.2743147739356417
4 1.665520656920488
5 1.9885282748550743
6 1.4519266258708554
7 1.6142529511961898
8 1.186511732710554
9 1.1214316204160948
10 0.7954960781032768
11 0.5891422565629302
12 0.4141817669610057
13 0.3320380439269659
14 0.2971354951759133
15 0.27485557982096376
16 0.2561263523633577
17 0.23947476927324007
18 0.22454042828186693
19 0.21108865517984543
20 0.19892666814240023
21 0.18789228535096253
22 0.17784809060885204
23 0.16867696743574462
24 0.1602785894468669
25 0.15256661689915535
26 0.14546643131347697
27 0.1389132884104582
28 0.13285080120083828
29 0.12722968664975792
30 0.12200672464894356
31 0.11714388924196442
32 0.11260762045479075
33 0.10836821150501547
34 0.10439929113676147
35 0.10067738472297143
36 0.09718154085301883
37 0.09389301257062203
38 0.09079498438524071
39 0.08787233775531293
40 0.0851114490146623
41 0.08250001474654041
42 0.08002690045156359
43 0.07768200904427103
44 0.075456166278088

In [28]:
ypred

[Value(data = -0.943948077894963),
 Value(data = 0.9137636459268738),
 Value(data = 0.9162886210825085),
 Value(data = -0.9022257383850496)]